# Project 4 — Model Optimization (Making Models Fast)

**Objective:** A model that's accurate but slow is useless in production. Convert the ResNet50 from Project 3 to ONNX format and benchmark the latency and size improvement.

**Input:** `resnet50_pneumonia.pth` (the trained model saved at the end of Project 3).
**Output:** `resnet50_pneumonia.onnx` + a Markdown report with before/after latency and size numbers.

In [1]:
import sys
!{sys.executable} -m pip install onnx onnxruntime

   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
    --------------------------------------- 0.3/17.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/17.2 MB 1.9 MB/s eta 0:00:09
   --- ------------------------------------ 1.3/17.2 MB 2.0 MB/s eta 0:00:09
   ---- ----------------------------------- 2.1/17.2 MB 2.3 MB/s eta 0:00:07
   ------ --------------------------------- 2.9/17.2 MB 2.7 MB/s eta 0:00:06
   ------- -------------------------------- 3.1/17.2 MB 2.8 MB/s eta 0:00:06
   --------- ------------------------------ 4.2/17.2 MB 2.7 MB/s eta 0:00:05
   ---------- ----------------------------- 4.7/17.2 MB 2.8 MB/s eta 0:00:05
   ------------ --------------------------- 5.2/17.2 MB 2.8 MB/s eta 0:00:05
   -------------- ------------------------- 6.0/17.2 MB 2.8 MB/s eta 0:00:05
   --------------- --------

In [2]:
import os
import time

import numpy as np
import onnxruntime as ort
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cpu")  # Benchmark on CPU -- this is what most production inference servers use by default

## 1. Load the trained PyTorch model

Rebuild the same ResNet50 architecture used in Project 3, then load the fine-tuned weights back in.

In [3]:
PYTORCH_MODEL_PATH = "resnet50_pneumonia.pth"
ONNX_MODEL_PATH = "resnet50_pneumonia.onnx"

model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)  # NORMAL vs PNEUMONIA
model.load_state_dict(torch.load(PYTORCH_MODEL_PATH, map_location=device))
model.eval()

print("PyTorch model loaded and set to eval mode.")

PyTorch model loaded and set to eval mode.


## 2. Export to ONNX

ONNX is a framework-neutral model format. Once exported, the model can run through a lightweight runtime (`onnxruntime`) instead of the full PyTorch library, which is typically faster for inference and easier to deploy.

In [4]:
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_MODEL_PATH,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=13,
    dynamo=False,  # use the classic TorchScript-based exporter (more stable, fewer extra dependencies)
)

print(f"Exported ONNX model to {ONNX_MODEL_PATH}")

C:\Users\jesut\AppData\Local\Temp\ipykernel_4360\452409677.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported ONNX model to resnet50_pneumonia.onnx


## 3. Sanity check — do both models agree?

Before trusting any speed comparison, confirm the ONNX model actually produces (near) identical predictions to the original PyTorch model on the same input.

In [5]:
ort_session = ort.InferenceSession(ONNX_MODEL_PATH, providers=["CPUExecutionProvider"])

test_input = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    pytorch_output = model(test_input).numpy()

onnx_output = ort_session.run(None, {"input": test_input.numpy()})[0]

max_diff = np.abs(pytorch_output - onnx_output).max()
print(f"Max difference between PyTorch and ONNX outputs: {max_diff:.6f}")
assert max_diff < 1e-3, "ONNX output diverges too much from PyTorch output!"
print("ONNX model output matches PyTorch model output.")

Max difference between PyTorch and ONNX outputs: 0.000002
ONNX model output matches PyTorch model output.


## 4. Benchmark inference latency

Run both models on the same batch of random inputs, many times, and measure average latency per single prediction. A few warm-up runs are excluded from timing, since the first run of any model is always artificially slow (memory allocation, graph optimization, etc.) -- including it would unfairly bias the comparison.

In [6]:
N_WARMUP = 5
N_RUNS = 50

benchmark_input = torch.randn(1, 3, 224, 224)
benchmark_input_np = benchmark_input.numpy()

# --- Warm-up (not timed) ---
with torch.no_grad():
    for _ in range(N_WARMUP):
        model(benchmark_input)
for _ in range(N_WARMUP):
    ort_session.run(None, {"input": benchmark_input_np})

# --- Time PyTorch ---
pytorch_times = []
with torch.no_grad():
    for _ in range(N_RUNS):
        start = time.perf_counter()
        model(benchmark_input)
        pytorch_times.append((time.perf_counter() - start) * 1000)  # ms

# --- Time ONNX Runtime ---
onnx_times = []
for _ in range(N_RUNS):
    start = time.perf_counter()
    ort_session.run(None, {"input": benchmark_input_np})
    onnx_times.append((time.perf_counter() - start) * 1000)  # ms

pytorch_avg_ms = np.mean(pytorch_times)
onnx_avg_ms = np.mean(onnx_times)
speedup_pct = (1 - onnx_avg_ms / pytorch_avg_ms) * 100

print(f"PyTorch average latency: {pytorch_avg_ms:.2f} ms")
print(f"ONNX average latency:    {onnx_avg_ms:.2f} ms")
print(f"Latency reduction:       {speedup_pct:.1f}%")

PyTorch average latency: 365.99 ms
ONNX average latency:    99.27 ms
Latency reduction:       72.9%


## 5. Compare file sizes

In [7]:
pytorch_size_mb = os.path.getsize(PYTORCH_MODEL_PATH) / (1024 * 1024)
onnx_size_mb = os.path.getsize(ONNX_MODEL_PATH) / (1024 * 1024)
size_change_pct = (1 - onnx_size_mb / pytorch_size_mb) * 100

print(f"PyTorch model size: {pytorch_size_mb:.2f} MB")
print(f"ONNX model size:    {onnx_size_mb:.2f} MB")
print(f"Size change:        {size_change_pct:.1f}%")

PyTorch model size: 89.99 MB
ONNX model size:    89.61 MB
Size change:        0.4%


## 6. Generate the Markdown report

This is the required deliverable — a short, standalone document stating the before/after numbers, matching the checklist's example format.

In [8]:
report = f"""# Optimization Results

## Objective
Convert the fine-tuned ResNet50 pneumonia classifier (Project 3) to ONNX format and measure the impact on inference latency and model size.

## Method
- Original model: PyTorch ResNet50 (`resnet50_pneumonia.pth`), fine-tuned for binary chest X-ray classification (NORMAL vs PNEUMONIA).
- Optimized model: same weights exported to ONNX (`resnet50_pneumonia.onnx`) via `torch.onnx.export`, opset 13.
- Benchmark: {N_RUNS} single-image inference calls on CPU, after {N_WARMUP} untimed warm-up runs, using identical random input tensors for both models.
- Correctness check: outputs from both models compared directly; max absolute difference was {max_diff:.6f} (effectively identical).

## Results

| Metric | PyTorch (original) | ONNX (optimized) | Change |
|---|---|---|---|
| Avg. inference latency | {pytorch_avg_ms:.1f} ms | {onnx_avg_ms:.1f} ms | {speedup_pct:.1f}% faster |
| Model file size | {pytorch_size_mb:.1f} MB | {onnx_size_mb:.1f} MB | {size_change_pct:.1f}% |

**Summary:** Reduced inference time from {pytorch_avg_ms:.0f}ms to {onnx_avg_ms:.0f}ms using ONNX ({speedup_pct:.1f}% latency reduction), with output predictions matching the original PyTorch model.

## Notes
- Benchmarked on CPU, single-image batch size. Real-world gains would likely be larger under concurrent/batched load, which ONNX Runtime handles more efficiently than eager-mode PyTorch.
- No accuracy was lost in this conversion -- ONNX export preserves the exact learned weights; only the execution engine changes.
"""

with open("optimization_results.md", "w") as f:
    f.write(report)

print(report)


# Optimization Results

## Objective
Convert the fine-tuned ResNet50 pneumonia classifier (Project 3) to ONNX format and measure the impact on inference latency and model size.

## Method
- Original model: PyTorch ResNet50 (`resnet50_pneumonia.pth`), fine-tuned for binary chest X-ray classification (NORMAL vs PNEUMONIA).
- Optimized model: same weights exported to ONNX (`resnet50_pneumonia.onnx`) via `torch.onnx.export`, opset 13.
- Benchmark: 50 single-image inference calls on CPU, after 5 untimed warm-up runs, using identical random input tensors for both models.
- Correctness check: outputs from both models compared directly; max absolute difference was 0.000002 (effectively identical).

## Results

| Metric | PyTorch (original) | ONNX (optimized) | Change |
|---|---|---|---|
| Avg. inference latency | 366.0 ms | 99.3 ms | 72.9% faster |
| Model file size | 90.0 MB | 89.6 MB | 0.4% |

**Summary:** Reduced inference time from 366ms to 99ms using ONNX (72.9% latency reduction), with o